In [ ]:
import os
import json
import requests
from pathlib import Path

# Root folder containing your SQL files
CONFIG_ROOT = "/Workspace/Repos/filipshahanski@gmail.com/dbx-ingestion-framework/config"
job_folder = "/Workspace/Repos/filipshahanski@gmail.com/dbx-ingestion-framework/config/jobs"

execution_order = [
    "master_control",
    "tables/config",
    "tables/bronze",
    "tables/silver",
    "tables/gold"
]

host = spark.conf.get("spark.databricks.workspaceUrl")

token = dbutils.notebook.entry_point.getDbutils() \
    .notebook().getContext().apiToken().get()

headers = {
    "Authorization": f"Bearer {token}"
}

In [ ]:


def execute_sql_file(file_path: Path):
    """Execute all SQL statements in a file."""
    print(f"Executing {file_path}")

    sql_text = file_path.read_text(encoding="utf-8").strip()

    if not sql_text:
        print(f"Skipping empty file: {file_path.name}")
        return

    spark.sql(sql_text)

    print(f"Completed {file_path.name}")


for folder in execution_order:
    folder_path = Path(CONFIG_ROOT) / folder

    if not folder_path.exists():
        print(f"Skipping missing folder: {folder_path}")
        continue

    # Execute files alphabetically
    for sql_file in sorted(folder_path.glob("*.sql")):
        execute_sql_file(sql_file)

print("All SQL scripts executed successfully.")

In [ ]:
existing_jobs = requests.get(
    f"https://{host}/api/2.2/jobs/list",
    headers=headers
).json().get("jobs", [])

existing_job_names = {
    j["settings"]["name"]: j["job_id"]
    for j in existing_jobs
}

In [ ]:


files = [f.path.replace("dbfs:", "") for f in dbutils.fs.ls(job_folder) if f.path.endswith(".json")]

In [ ]:
for file_path in files:
    print(f"Processing {file_path}")

    # read file
    local_path = file_path.replace("/Workspace", "/dbfs")
    
    with open(file_path, "r") as f:
        job_def = json.load(f)

    job_name = job_def["name"]

    # check if exists
    if job_name in existing_job_names:
        print(f"Skipping (exists): {job_name}")
        continue

    # create job
    response = requests.post(
        f"https://{host}/api/2.2/jobs/create",
        headers=headers,
        json=job_def
    )

    print(f"Created: {job_name} -> {response.status_code}")